# 07 — Transformation-Based Synthetic-Data Generation and Validation

This notebook creates and validates the first synthetic-augmentation dataset for
limited-data drone–bird micro-Doppler classification.

The generator uses only the balanced **10% real training subset** (575 bird and
575 drone samples). It produces one traceable synthetic child per real parent.
The official validation and test partitions remain entirely real and are never
used for generation, filtering, or parameter selection.

This is a **transformation-based augmentation baseline**, not an independent
physics simulator or generative neural network. Classification with the accepted
dataset is performed in Notebook 08.

## 1. Experimental Protocol

The generator applies mild, label-preserving transformations:

- translation of at most five bins along the 150-bin feature axis;
- amplitude and contrast scaling between 0.90 and 1.10;
- Gaussian noise with standard deviation between 0.002 and 0.015;
- a 30% probability of masking a two-to-six-bin feature interval;
- final clipping to the preprocessing interval `[0,1]`.

Each synthetic observation stores its parent index and transformation parameters.
Quality is assessed through structural assertions, parent similarity before and
after translation alignment, class-level descriptive statistics, paired visual
inspection, PCA overlap, clipping rates, and targeted inspection of outliers.

The complete generation uses its own seed-42 random generator, making it
independent of diagnostic cell execution. Saved files are protected against
accidental overwriting during a normal top-to-bottom rerun.

## 2. Imports and Reproducible Configuration

In [ ]:
import hashlib
import json
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from pathlib import Path
from sklearn.decomposition import PCA

In [ ]:
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

SOURCE_SUBSET = "10_percent"
SYNTHETIC_RATIO = 1.0

# The validated files already exist. Keep this False during normal reruns.
SAVE_GENERATED_DATA = False
ALLOW_OVERWRITE = False

GENERATION_NAME = (
    f"{SOURCE_SUBSET}_signal_augmentation_"
    f"ratio_{SYNTHETIC_RATIO:g}_seed_{RANDOM_SEED}"
)

print("Random seed:", RANDOM_SEED)
print("Source subset:", SOURCE_SUBSET)
print("Synthetic-to-real ratio:", SYNTHETIC_RATIO)
print("Generation name:", GENERATION_NAME)
print("Save generated data:", SAVE_GENERATED_DATA)

## 3. Project Paths and Input Integrity

In [ ]:
OFFICIAL_DATA_DIR = Path(
    "../data/processed/official_split"
)

LIMITED_DATA_DIR = Path(
    "../data/processed/limited_subsets"
)

SYNTHETIC_DATA_DIR = Path(
    "../data/processed/synthetic_subsets"
)

SYNTHETIC_OUTPUT_DIR = Path(
    "../outputs/synthetic_data_validation"
) / GENERATION_NAME

SYNTHETIC_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SYNTHETIC_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

required_files = [
    OFFICIAL_DATA_DIR / "X_train.npy",
    OFFICIAL_DATA_DIR / "y_train.npy",
    OFFICIAL_DATA_DIR / "metadata_train.csv",
    LIMITED_DATA_DIR / "indices_10_percent.npy",
    LIMITED_DATA_DIR / "subset_manifest.json"
]

missing_files = [
    path for path in required_files
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "Missing required files:\n"
        + "\n".join(
            str(path.resolve())
            for path in missing_files
        )
    )

print("All required files were found.")
print("Synthetic-data directory:", SYNTHETIC_DATA_DIR.resolve())
print("Validation-output directory:", SYNTHETIC_OUTPUT_DIR.resolve())

## 4. Load and Validate the 10% Real Source Subset

In [ ]:
X_train_complete = np.load(
    OFFICIAL_DATA_DIR / "X_train.npy",
    mmap_mode="r"
)

y_train_complete = np.load(
    OFFICIAL_DATA_DIR / "y_train.npy",
    mmap_mode="r"
)

metadata_train_complete = pd.read_csv(
    OFFICIAL_DATA_DIR / "metadata_train.csv"
)

source_indices = np.load(
    LIMITED_DATA_DIR / "indices_10_percent.npy"
)

X_real_source = np.asarray(
    X_train_complete[source_indices],
    dtype=np.float32
)

y_real_source = np.asarray(
    y_train_complete[source_indices],
    dtype=np.uint8
)

metadata_real_source = (
    metadata_train_complete
    .iloc[source_indices]
    .copy()
    .reset_index(drop=True)
)

metadata_real_source[
    "official_training_index"
] = source_indices

print("Source tensor shape:", X_real_source.shape)
print("Source label shape:", y_real_source.shape)
print(
    "Class counts [bird, drone]:",
    np.bincount(y_real_source)
)
print(
    "Value range:",
    float(X_real_source.min()),
    "to",
    float(X_real_source.max())
)

In [ ]:
assert X_real_source.shape == (
    1150,
    5,
    150
)

assert y_real_source.shape == (
    1150,
)

assert len(metadata_real_source) == 1150

assert np.array_equal(
    np.bincount(y_real_source),
    [575, 575]
)

assert len(source_indices) == len(
    np.unique(source_indices)
)

assert source_indices.min() >= 0

assert source_indices.max() < len(
    y_train_complete
)

assert np.isfinite(
    X_real_source
).all()

assert X_real_source.min() >= 0.0
assert X_real_source.max() <= 1.0

assert np.array_equal(
    y_real_source,
    np.asarray(
        y_train_complete[source_indices],
        dtype=np.uint8
    )
)

print(
    "The 10% real source subset passed "
    "all validation checks."
)

## 5. Controlled Signal-Domain Generator

Translations use nearest-edge filling rather than circular wrapping. This avoids
moving content from one edge of the representation to the opposite edge. All
operations retain the original `(5,150)` shape and parent label.

In [ ]:
def shift_feature_axis(sample, shift):
    """
    Translate a (5, 150) sample along its second axis.

    Empty positions are filled using the nearest boundary value.
    Circular wrapping is intentionally avoided.
    """
    if shift == 0:
        return sample.copy()

    shifted = np.empty_like(sample)

    if shift > 0:
        shifted[:, :shift] = sample[:, :1]
        shifted[:, shift:] = sample[:, :-shift]
    else:
        width = abs(shift)
        shifted[:, -width:] = sample[:, -1:]
        shifted[:, :-width] = sample[:, width:]

    return shifted

In [ ]:
def augment_radar_sample(sample, generator):
    """
    Create one synthetic radar sample using mild,
    label-preserving transformations.

    Parameters
    ----------
    sample : np.ndarray
        Normalized radar tensor with shape (5, 150).

    generator : np.random.Generator
        Seeded NumPy random generator.

    Returns
    -------
    synthetic_sample : np.ndarray
        Augmented float32 tensor with shape (5, 150).

    parameters : dict
        Transformation parameters used to create the sample.
    """
    synthetic_sample = np.asarray(
        sample,
        dtype=np.float32
    ).copy()

    parameters = {}

    # 1. Small translation along the 150-bin feature axis.
    feature_shift = int(
        generator.integers(-5, 6)
    )

    synthetic_sample = shift_feature_axis(
        synthetic_sample,
        feature_shift
    )

    parameters["feature_shift"] = feature_shift

    # 2. Mild amplitude scaling.
    amplitude_scale = float(
        generator.uniform(0.90, 1.10)
    )

    synthetic_sample *= amplitude_scale

    parameters["amplitude_scale"] = (
        amplitude_scale
    )

    # 3. Mild contrast modification around the sample mean.
    contrast_factor = float(
        generator.uniform(0.90, 1.10)
    )

    sample_mean = float(
        synthetic_sample.mean()
    )

    synthetic_sample = (
        sample_mean
        + contrast_factor
        * (synthetic_sample - sample_mean)
    )

    parameters["contrast_factor"] = (
        contrast_factor
    )

    # 4. Low-intensity additive Gaussian noise.
    noise_standard_deviation = float(
        generator.uniform(0.002, 0.015)
    )

    noise = generator.normal(
        loc=0.0,
        scale=noise_standard_deviation,
        size=synthetic_sample.shape
    ).astype(np.float32)

    synthetic_sample += noise

    parameters[
        "noise_standard_deviation"
    ] = noise_standard_deviation

    # 5. Optional short mask along the 150-bin axis.
    apply_mask = bool(
        generator.random() < 0.30
    )

    mask_start = -1
    mask_width = 0

    if apply_mask:
        mask_width = int(
            generator.integers(2, 7)
        )

        mask_start = int(
            generator.integers(
                0,
                synthetic_sample.shape[1]
                - mask_width + 1
            )
        )

        local_fill_value = float(
            synthetic_sample.mean()
        )

        synthetic_sample[
            :,
            mask_start:
            mask_start + mask_width
        ] = local_fill_value

    parameters["mask_applied"] = apply_mask
    parameters["mask_start"] = mask_start
    parameters["mask_width"] = mask_width

    # Preserve the preprocessing interval.
    synthetic_sample = np.clip(
        synthetic_sample,
        0.0,
        1.0
    ).astype(np.float32)

    return synthetic_sample, parameters

### 5.1 Single-Sample Functional Test

In [ ]:
test_rng = np.random.default_rng(RANDOM_SEED)

test_synthetic_sample, test_parameters = (
    augment_radar_sample(
        X_real_source[0],
        test_rng
    )
)

print(
    "Original shape:",
    X_real_source[0].shape
)

print(
    "Synthetic shape:",
    test_synthetic_sample.shape
)

print(
    "Synthetic dtype:",
    test_synthetic_sample.dtype
)

print(
    "Synthetic range:",
    float(test_synthetic_sample.min()),
    "to",
    float(test_synthetic_sample.max())
)

print(
    "Transformation parameters:",
    test_parameters
)

assert (
    test_synthetic_sample.shape
    == X_real_source[0].shape
)

assert (
    test_synthetic_sample.dtype
    == np.float32
)

assert np.isfinite(
    test_synthetic_sample
).all()

assert test_synthetic_sample.min() >= 0.0
assert test_synthetic_sample.max() <= 1.0

assert not np.array_equal(
    test_synthetic_sample,
    X_real_source[0]
)

print("Single-sample augmentation test passed.")

### 5.2 Generate the Balanced Synthetic Dataset

In [ ]:
generation_rng = np.random.default_rng(RANDOM_SEED)

synthetic_samples = []
synthetic_labels = []
generation_records = []

synthetic_samples_per_real = int(
    SYNTHETIC_RATIO
)

assert SYNTHETIC_RATIO == float(
    synthetic_samples_per_real
), (
    "The initial generator expects an "
    "integer synthetic-to-real ratio."
)

synthetic_id = 0

for local_parent_index, (
    real_sample,
    real_label
) in enumerate(
    zip(
        X_real_source,
        y_real_source
    )
):
    for child_number in range(
        synthetic_samples_per_real
    ):
        synthetic_sample, parameters = (
            augment_radar_sample(
                real_sample,
                generation_rng
            )
        )

        synthetic_samples.append(
            synthetic_sample
        )

        synthetic_labels.append(
            real_label
        )

        generation_records.append({
            "synthetic_index": synthetic_id,
            "local_parent_index":
                local_parent_index,
            "official_training_index": int(
                source_indices[
                    local_parent_index
                ]
            ),
            "parent_label": int(real_label),
            "parent_target_group": (
                "drone"
                if real_label == 1
                else "bird"
            ),
            "child_number": child_number,
            **parameters
        })

        synthetic_id += 1

X_synthetic = np.stack(
    synthetic_samples
).astype(np.float32)

y_synthetic = np.asarray(
    synthetic_labels,
    dtype=np.uint8
)

synthetic_metadata = pd.DataFrame(
    generation_records
)

print(
    "Synthetic tensor shape:",
    X_synthetic.shape
)

print(
    "Synthetic label shape:",
    y_synthetic.shape
)

print(
    "Class counts [bird, drone]:",
    np.bincount(y_synthetic)
)

display(
    synthetic_metadata.head()
)

In [ ]:
expected_synthetic_samples = int(
    len(X_real_source)
    * SYNTHETIC_RATIO
)

assert X_synthetic.shape == (
    expected_synthetic_samples,
    5,
    150
)

assert y_synthetic.shape == (
    expected_synthetic_samples,
)

assert len(
    synthetic_metadata
) == expected_synthetic_samples

assert np.array_equal(
    np.bincount(y_synthetic),
    [
        expected_synthetic_samples // 2,
        expected_synthetic_samples // 2
    ]
)

assert np.isfinite(
    X_synthetic
).all()

assert X_synthetic.min() >= 0.0
assert X_synthetic.max() <= 1.0

assert X_synthetic.dtype == np.float32
assert y_synthetic.dtype == np.uint8

assert np.array_equal(
    y_synthetic,
    synthetic_metadata[
        "parent_label"
    ].to_numpy(dtype=np.uint8)
)

assert (
    synthetic_metadata[
        "official_training_index"
    ]
    .isin(source_indices)
    .all()
)

assert not np.any(
    np.all(
        X_synthetic == X_real_source,
        axis=(1, 2)
    )
)

print(
    "The generated synthetic dataset "
    "passed all structural checks."
)

print(
    "Synthetic samples:",
    len(X_synthetic)
)

print(
    "Synthetic birds:",
    int(np.sum(y_synthetic == 0))
)

print(
    "Synthetic drones:",
    int(np.sum(y_synthetic == 1))
)

print(
    "Value range:",
    float(X_synthetic.min()),
    "to",
    float(X_synthetic.max())
)

### 5.3 Structural Validation Result

The generator produced 1,150 finite `float32` tensors in `[0,1]`, with 575
samples per class. Every label matches its real training parent, every parent
index belongs to the approved 10% subset, and no synthetic child is an exact
copy of its corresponding real parent.

## 6. Transformation and Parent-Similarity Analysis

In [ ]:
transformation_summary = pd.DataFrame({
    "parameter": [
        "Feature shift",
        "Amplitude scale",
        "Contrast factor",
        "Noise standard deviation",
        "Mask applied"
    ],
    "minimum": [
        synthetic_metadata["feature_shift"].min(),
        synthetic_metadata["amplitude_scale"].min(),
        synthetic_metadata["contrast_factor"].min(),
        synthetic_metadata[
            "noise_standard_deviation"
        ].min(),
        synthetic_metadata["mask_applied"].min()
    ],
    "mean": [
        synthetic_metadata["feature_shift"].mean(),
        synthetic_metadata["amplitude_scale"].mean(),
        synthetic_metadata["contrast_factor"].mean(),
        synthetic_metadata[
            "noise_standard_deviation"
        ].mean(),
        synthetic_metadata["mask_applied"].mean()
    ],
    "maximum": [
        synthetic_metadata["feature_shift"].max(),
        synthetic_metadata["amplitude_scale"].max(),
        synthetic_metadata["contrast_factor"].max(),
        synthetic_metadata[
            "noise_standard_deviation"
        ].max(),
        synthetic_metadata["mask_applied"].max()
    ]
})

display(
    transformation_summary.round(4)
)

print(
    "Masked samples:",
    int(
        synthetic_metadata[
            "mask_applied"
        ].sum()
    ),
    "/",
    len(synthetic_metadata)
)

In [ ]:
parent_samples = X_real_source[
    synthetic_metadata[
        "local_parent_index"
    ].to_numpy()
]

differences = (
    X_synthetic
    - parent_samples
)

parent_mae = np.mean(
    np.abs(differences),
    axis=(1, 2)
)

parent_rmse = np.sqrt(
    np.mean(
        differences ** 2,
        axis=(1, 2)
    )
)

parent_relative_l2 = (
    np.linalg.norm(
        differences.reshape(
            len(differences),
            -1
        ),
        axis=1
    )
    /
    (
        np.linalg.norm(
            parent_samples.reshape(
                len(parent_samples),
                -1
            ),
            axis=1
        )
        + 1e-8
    )
)

parent_correlations = []

for real_sample, synthetic_sample in zip(
    parent_samples,
    X_synthetic
):
    real_flat = real_sample.reshape(-1)
    synthetic_flat = synthetic_sample.reshape(-1)

    if (
        np.std(real_flat) == 0
        or np.std(synthetic_flat) == 0
    ):
        correlation = np.nan
    else:
        correlation = np.corrcoef(
            real_flat,
            synthetic_flat
        )[0, 1]

    parent_correlations.append(
        correlation
    )

parent_correlations = np.asarray(
    parent_correlations,
    dtype=np.float32
)

clipped_fraction = np.mean(
    (X_synthetic == 0.0)
    | (X_synthetic == 1.0),
    axis=(1, 2)
)

similarity_metrics_df = pd.DataFrame({
    "synthetic_index":
        synthetic_metadata["synthetic_index"],
    "parent_label":
        synthetic_metadata["parent_label"],
    "parent_target_group":
        synthetic_metadata[
            "parent_target_group"
        ],
    "mae_to_parent": parent_mae,
    "rmse_to_parent": parent_rmse,
    "relative_l2_to_parent":
        parent_relative_l2,
    "correlation_with_parent":
        parent_correlations,
    "clipped_fraction":
        clipped_fraction
})

similarity_summary = (
    similarity_metrics_df
    .groupby(
        "parent_target_group",
        observed=True
    )
    .agg(
        samples=(
            "synthetic_index",
            "size"
        ),
        mean_mae=(
            "mae_to_parent",
            "mean"
        ),
        median_mae=(
            "mae_to_parent",
            "median"
        ),
        mean_rmse=(
            "rmse_to_parent",
            "mean"
        ),
        mean_relative_l2=(
            "relative_l2_to_parent",
            "mean"
        ),
        mean_parent_correlation=(
            "correlation_with_parent",
            "mean"
        ),
        median_parent_correlation=(
            "correlation_with_parent",
            "median"
        ),
        mean_clipped_fraction=(
            "clipped_fraction",
            "mean"
        )
    )
    .reset_index()
)

display(
    similarity_summary.round(4)
)

In [ ]:
fig, axes = plt.subplots(
    1,
    3,
    figsize=(16, 4),
    constrained_layout=True
)

sns.histplot(
    data=similarity_metrics_df,
    x="mae_to_parent",
    hue="parent_target_group",
    bins=30,
    element="step",
    stat="density",
    common_norm=False,
    ax=axes[0]
)

axes[0].set_title(
    "Absolute Difference from Parent"
)
axes[0].set_xlabel("Mean absolute error")

sns.histplot(
    data=similarity_metrics_df,
    x="correlation_with_parent",
    hue="parent_target_group",
    bins=30,
    element="step",
    stat="density",
    common_norm=False,
    ax=axes[1]
)

axes[1].set_title(
    "Real–Synthetic Parent Correlation"
)
axes[1].set_xlabel("Pearson correlation")

sns.histplot(
    data=similarity_metrics_df,
    x="clipped_fraction",
    hue="parent_target_group",
    bins=30,
    element="step",
    stat="density",
    common_norm=False,
    ax=axes[2]
)

axes[2].set_title(
    "Fraction of Values at 0 or 1"
)
axes[2].set_xlabel("Clipped-value fraction")

plt.show()

### 6.1 Initial Similarity Interpretation

Global changes are mild, but unaligned pixel correlation is lower for translated
samples—especially birds with narrow peaks. Because ordinary correlation is not
translation-invariant, alignment-aware similarity is evaluated before drawing a
quality conclusion.

## 7. Real–Synthetic Distribution Validation

In [ ]:
def calculate_sample_statistics(
    samples,
    labels,
    source_name
):
    records = []

    for label_value, target_group in [
        (0, "bird"),
        (1, "drone")
    ]:
        class_samples = samples[
            labels == label_value
        ]

        per_sample_mean = np.mean(
            class_samples,
            axis=(1, 2)
        )

        per_sample_standard_deviation = (
            np.std(
                class_samples,
                axis=(1, 2)
            )
        )

        per_sample_energy = np.mean(
            class_samples ** 2,
            axis=(1, 2)
        )

        records.append({
            "source": source_name,
            "target_group": target_group,
            "samples": len(class_samples),
            "mean_intensity":
                per_sample_mean.mean(),
            "median_intensity":
                np.median(per_sample_mean),
            "mean_standard_deviation":
                per_sample_standard_deviation.mean(),
            "mean_energy":
                per_sample_energy.mean(),
            "minimum_value":
                class_samples.min(),
            "maximum_value":
                class_samples.max()
        })

    return records


distribution_records = []

distribution_records.extend(
    calculate_sample_statistics(
        X_real_source,
        y_real_source,
        "real"
    )
)

distribution_records.extend(
    calculate_sample_statistics(
        X_synthetic,
        y_synthetic,
        "synthetic"
    )
)

distribution_summary_df = pd.DataFrame(
    distribution_records
)

display(
    distribution_summary_df.round(4)
)

In [ ]:
sample_distribution_records = []

for source_name, samples, labels in [
    (
        "Real",
        X_real_source,
        y_real_source
    ),
    (
        "Synthetic",
        X_synthetic,
        y_synthetic
    )
]:
    for sample_index, (
        sample,
        label
    ) in enumerate(
        zip(samples, labels)
    ):
        sample_distribution_records.append({
            "source": source_name,
            "target_group": (
                "Drone"
                if label == 1
                else "Bird"
            ),
            "sample_index": sample_index,
            "mean_intensity":
                float(sample.mean()),
            "standard_deviation":
                float(sample.std()),
            "energy":
                float(np.mean(sample ** 2))
        })

sample_distribution_df = pd.DataFrame(
    sample_distribution_records
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(16, 5),
    constrained_layout=True
)

for axis, metric, title in zip(
    axes,
    [
        "mean_intensity",
        "standard_deviation",
        "energy"
    ],
    [
        "Mean Intensity",
        "Within-Sample Standard Deviation",
        "Mean-Squared Energy"
    ]
):
    sns.boxplot(
        data=sample_distribution_df,
        x="target_group",
        y=metric,
        hue="source",
        showfliers=False,
        ax=axis
    )

    axis.set_title(title)
    axis.set_xlabel("True target group")
    axis.grid(
        axis="y",
        alpha=0.2
    )

plt.show()

In [ ]:
bird_example_indices = np.where(
    y_synthetic == 0
)[0][:3]

drone_example_indices = np.where(
    y_synthetic == 1
)[0][:3]

visual_example_indices = np.concatenate([
    bird_example_indices,
    drone_example_indices
])

fig, axes = plt.subplots(
    len(visual_example_indices),
    3,
    figsize=(16, 14),
    constrained_layout=True
)

for row, synthetic_index in enumerate(
    visual_example_indices
):
    parent_index = int(
        synthetic_metadata.loc[
            synthetic_index,
            "local_parent_index"
        ]
    )

    target_group = (
        synthetic_metadata.loc[
            synthetic_index,
            "parent_target_group"
        ]
    )

    real_sample = X_real_source[
        parent_index
    ]

    synthetic_sample = X_synthetic[
        synthetic_index
    ]

    absolute_difference = np.abs(
        synthetic_sample
        - real_sample
    )

    axes[row, 0].imshow(
        real_sample,
        aspect="auto",
        origin="lower",
        cmap="viridis",
        vmin=0,
        vmax=1
    )

    axes[row, 1].imshow(
        synthetic_sample,
        aspect="auto",
        origin="lower",
        cmap="viridis",
        vmin=0,
        vmax=1
    )

    axes[row, 2].imshow(
        absolute_difference,
        aspect="auto",
        origin="lower",
        cmap="magma",
        vmin=0,
        vmax=max(
            0.10,
            float(
                absolute_difference.max()
            )
        )
    )

    axes[row, 0].set_ylabel(
        f"{target_group.title()}\n"
        f"Pair {synthetic_index}"
    )

    if row == 0:
        axes[row, 0].set_title(
            "Real Parent"
        )
        axes[row, 1].set_title(
            "Synthetic Child"
        )
        axes[row, 2].set_title(
            "Absolute Difference"
        )

    for axis in axes[row]:
        axis.set_xlabel(
            "Feature-axis bin"
        )
        axis.set_yticks(
            range(real_sample.shape[0])
        )

plt.show()

### 7.1 Distribution and Visual Findings

Real and synthetic intensity, within-sample variability, and energy distributions
are nearly identical within each class. Paired images preserve the dominant
micro-Doppler structures while introducing visible but controlled local changes.

## 8. Translation-Aware Similarity Validation

In [ ]:
aligned_synthetic_samples = []

for synthetic_index in range(
    len(X_synthetic)
):
    feature_shift = int(
        synthetic_metadata.loc[
            synthetic_index,
            "feature_shift"
        ]
    )

    aligned_sample = shift_feature_axis(
        X_synthetic[synthetic_index],
        -feature_shift
    )

    aligned_synthetic_samples.append(
        aligned_sample
    )

X_synthetic_aligned = np.stack(
    aligned_synthetic_samples
).astype(np.float32)

aligned_differences = (
    X_synthetic_aligned
    - parent_samples
)

aligned_mae = np.mean(
    np.abs(aligned_differences),
    axis=(1, 2)
)

aligned_rmse = np.sqrt(
    np.mean(
        aligned_differences ** 2,
        axis=(1, 2)
    )
)

aligned_correlations = []

for real_sample, aligned_sample in zip(
    parent_samples,
    X_synthetic_aligned
):
    real_flat = real_sample.reshape(-1)
    aligned_flat = aligned_sample.reshape(-1)

    if (
        np.std(real_flat) == 0
        or np.std(aligned_flat) == 0
    ):
        correlation = np.nan
    else:
        correlation = np.corrcoef(
            real_flat,
            aligned_flat
        )[0, 1]

    aligned_correlations.append(
        correlation
    )

aligned_correlations = np.asarray(
    aligned_correlations,
    dtype=np.float32
)

similarity_metrics_df[
    "aligned_mae"
] = aligned_mae

similarity_metrics_df[
    "aligned_rmse"
] = aligned_rmse

similarity_metrics_df[
    "aligned_parent_correlation"
] = aligned_correlations

In [ ]:
alignment_summary_df = (
    similarity_metrics_df
    .groupby(
        "parent_target_group",
        observed=True
    )
    .agg(
        samples=(
            "synthetic_index",
            "size"
        ),
        raw_mean_mae=(
            "mae_to_parent",
            "mean"
        ),
        aligned_mean_mae=(
            "aligned_mae",
            "mean"
        ),
        raw_mean_correlation=(
            "correlation_with_parent",
            "mean"
        ),
        aligned_mean_correlation=(
            "aligned_parent_correlation",
            "mean"
        ),
        aligned_median_correlation=(
            "aligned_parent_correlation",
            "median"
        ),
        minimum_aligned_correlation=(
            "aligned_parent_correlation",
            "min"
        )
    )
    .reset_index()
)

display(
    alignment_summary_df.round(4)
)

In [ ]:
alignment_plot_df = pd.concat([
    similarity_metrics_df[
        [
            "parent_target_group",
            "correlation_with_parent"
        ]
    ]
    .rename(
        columns={
            "correlation_with_parent":
                "correlation"
        }
    )
    .assign(comparison="Before alignment"),

    similarity_metrics_df[
        [
            "parent_target_group",
            "aligned_parent_correlation"
        ]
    ]
    .rename(
        columns={
            "aligned_parent_correlation":
                "correlation"
        }
    )
    .assign(comparison="After alignment")
], ignore_index=True)

plt.figure(figsize=(10, 6))

sns.boxplot(
    data=alignment_plot_df,
    x="parent_target_group",
    y="correlation",
    hue="comparison",
    showfliers=False
)

plt.axhline(
    0.80,
    color="black",
    linestyle="--",
    linewidth=1,
    label="0.80 reference"
)

plt.title(
    "Real–Synthetic Parent Correlation "
    "Before and After Translation Alignment"
)

plt.xlabel("Parent target group")
plt.ylabel("Pearson correlation")
plt.grid(axis="y", alpha=0.2)
plt.legend()
plt.show()

In [ ]:
similarity_metrics_df[
    "absolute_feature_shift"
] = (
    synthetic_metadata[
        "feature_shift"
    ]
    .abs()
    .to_numpy()
)

shift_effect_df = (
    similarity_metrics_df
    .groupby(
        [
            "parent_target_group",
            "absolute_feature_shift"
        ],
        observed=True
    )
    .agg(
        samples=(
            "synthetic_index",
            "size"
        ),
        raw_mean_correlation=(
            "correlation_with_parent",
            "mean"
        ),
        aligned_mean_correlation=(
            "aligned_parent_correlation",
            "mean"
        ),
        aligned_mean_mae=(
            "aligned_mae",
            "mean"
        )
    )
    .reset_index()
)

display(
    shift_effect_df.round(4)
)

### 8.1 Alignment Result

After reversing the known translations, mean parent correlation reaches 0.9818
for birds and 0.9874 for drones. The corresponding aligned MAE values are 0.0263
and 0.0214. The low raw bird correlation is therefore explained by translation
sensitivity rather than loss of signal structure.

## 9. PCA Overlap and Final Quality Indicators

In [ ]:
combined_samples = np.concatenate(
    [
        X_real_source,
        X_synthetic
    ],
    axis=0
)

combined_flat = combined_samples.reshape(
    len(combined_samples),
    -1
)

combined_source = np.concatenate([
    np.repeat(
        "Real",
        len(X_real_source)
    ),
    np.repeat(
        "Synthetic",
        len(X_synthetic)
    )
])

combined_labels = np.concatenate([
    y_real_source,
    y_synthetic
])

pca = PCA(
    n_components=2,
    random_state=RANDOM_SEED
)

pca_coordinates = pca.fit_transform(
    combined_flat
)

pca_df = pd.DataFrame({
    "principal_component_1":
        pca_coordinates[:, 0],
    "principal_component_2":
        pca_coordinates[:, 1],
    "source": combined_source,
    "target_group": np.where(
        combined_labels == 1,
        "Drone",
        "Bird"
    )
})

figure, axes = plt.subplots(
    1,
    2,
    figsize=(15, 6),
    constrained_layout=True
)

sns.scatterplot(
    data=pca_df,
    x="principal_component_1",
    y="principal_component_2",
    hue="source",
    alpha=0.45,
    s=25,
    ax=axes[0]
)

axes[0].set_title(
    "PCA Distribution by Data Source"
)

sns.scatterplot(
    data=pca_df,
    x="principal_component_1",
    y="principal_component_2",
    hue="target_group",
    style="source",
    alpha=0.45,
    s=25,
    ax=axes[1]
)

axes[1].set_title(
    "PCA Distribution by Class and Source"
)

for axis in axes:
    axis.grid(alpha=0.2)

plt.show()

print(
    "Variance explained by two components:",
    round(
        float(
            pca.explained_variance_ratio_.sum()
        ),
        4
    )
)

In [ ]:
quality_summary = pd.DataFrame({
    "quality_indicator": [
        "Bird mean aligned correlation",
        "Drone mean aligned correlation",
        "Bird median aligned correlation",
        "Drone median aligned correlation",
        "Samples below 0.80 aligned correlation",
        "Mean bird clipped fraction",
        "Mean drone clipped fraction",
        "PCA variance explained by two components"
    ],
    "value": [
        similarity_metrics_df.loc[
            similarity_metrics_df[
                "parent_target_group"
            ] == "bird",
            "aligned_parent_correlation"
        ].mean(),

        similarity_metrics_df.loc[
            similarity_metrics_df[
                "parent_target_group"
            ] == "drone",
            "aligned_parent_correlation"
        ].mean(),

        similarity_metrics_df.loc[
            similarity_metrics_df[
                "parent_target_group"
            ] == "bird",
            "aligned_parent_correlation"
        ].median(),

        similarity_metrics_df.loc[
            similarity_metrics_df[
                "parent_target_group"
            ] == "drone",
            "aligned_parent_correlation"
        ].median(),

        np.sum(
            similarity_metrics_df[
                "aligned_parent_correlation"
            ] < 0.80
        ),

        similarity_metrics_df.loc[
            similarity_metrics_df[
                "parent_target_group"
            ] == "bird",
            "clipped_fraction"
        ].mean(),

        similarity_metrics_df.loc[
            similarity_metrics_df[
                "parent_target_group"
            ] == "drone",
            "clipped_fraction"
        ].mean(),

        pca.explained_variance_ratio_.sum()
    ]
})

display(
    quality_summary.round(4)
)

low_correlation_samples = (
    similarity_metrics_df[
        similarity_metrics_df[
            "aligned_parent_correlation"
        ] < 0.80
    ]
    .merge(
        synthetic_metadata,
        on=[
            "synthetic_index",
            "parent_label",
            "parent_target_group"
        ],
        how="left"
    )
)

print(
    "Samples below 0.80 aligned correlation:",
    len(low_correlation_samples)
)

display(
    low_correlation_samples[
        [
            "synthetic_index",
            "parent_target_group",
            "official_training_index",
            "feature_shift",
            "mask_applied",
            "mask_width",
            "noise_standard_deviation",
            "aligned_parent_correlation"
        ]
    ].round(4)
)

### 9.1 PCA and Quality Assessment

The first two principal components explain 58.92% of the total variance. Real and
synthetic observations overlap substantially, and the synthetic data does not
form an isolated source-specific cluster. Only one of 1,150 samples falls below
the diagnostic aligned-correlation reference of 0.80.

### 9.2 Targeted Inspection of the Single Low-Correlation Sample

In [ ]:
outlier_synthetic_index = 162

outlier_parent_index = int(
    synthetic_metadata.loc[
        outlier_synthetic_index,
        "local_parent_index"
    ]
)

outlier_shift = int(
    synthetic_metadata.loc[
        outlier_synthetic_index,
        "feature_shift"
    ]
)

outlier_parent = X_real_source[
    outlier_parent_index
]

outlier_synthetic = X_synthetic[
    outlier_synthetic_index
]

outlier_aligned = (
    X_synthetic_aligned[
        outlier_synthetic_index
    ]
)

figure, axes = plt.subplots(
    1,
    4,
    figsize=(18, 4),
    constrained_layout=True
)

images = [
    outlier_parent,
    outlier_synthetic,
    outlier_aligned,
    np.abs(
        outlier_parent
        - outlier_aligned
    )
]

titles = [
    "Real Parent",
    (
        "Synthetic Child\n"
        f"Shift = {outlier_shift}"
    ),
    "Synthetic After Alignment",
    "Aligned Absolute Difference"
]

colour_maps = [
    "viridis",
    "viridis",
    "viridis",
    "magma"
]

for axis, image, title, colour_map in zip(
    axes,
    images,
    titles,
    colour_maps
):
    plot = axis.imshow(
        image,
        aspect="auto",
        origin="lower",
        cmap=colour_map,
        vmin=0,
        vmax=(
            1
            if colour_map == "viridis"
            else max(
                0.10,
                float(image.max())
            )
        )
    )

    axis.set_title(title)
    axis.set_xlabel("Feature-axis bin")
    axis.set_ylabel("Range-cell index")

    figure.colorbar(
        plot,
        ax=axis,
        fraction=0.046,
        pad=0.04
    )

plt.show()

display(
    synthetic_metadata.loc[
        [
            outlier_synthetic_index
        ]
    ]
)

display(
    similarity_metrics_df.loc[
        [
            outlier_synthetic_index
        ],
        [
            "parent_target_group",
            "mae_to_parent",
            "aligned_mae",
            "correlation_with_parent",
            "aligned_parent_correlation",
            "clipped_fraction"
        ]
    ].round(4)
)

### 9.3 Outlier Decision

Synthetic sample 162 has an aligned correlation of 0.7691 because its four-bin
mask overlaps the strongest bird peak after translation. The principal structure
remains visible, aligned MAE is only 0.0256, and just 0.4% of values are clipped.
It is retained as a plausible difficult augmentation. No similarity-based
filtering is applied, avoiding post-generation selection bias.

## 10. Persist the Accepted Dataset and Validation Evidence

In [ ]:
SYNTHETIC_DATASET_DIR = SYNTHETIC_DATA_DIR / GENERATION_NAME
SYNTHETIC_DATASET_DIR.mkdir(parents=True, exist_ok=True)

synthetic_files = {
    "features": SYNTHETIC_DATASET_DIR / "X_synthetic.npy",
    "labels": SYNTHETIC_DATASET_DIR / "y_synthetic.npy",
    "metadata": SYNTHETIC_DATASET_DIR / "metadata_synthetic.csv",
    "manifest": SYNTHETIC_DATASET_DIR / "generation_manifest.json"
}


def calculate_sha256(file_path):
    digest = hashlib.sha256()
    with open(file_path, "rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


existing_synthetic_files = [
    path for path in synthetic_files.values() if path.exists()
]

if SAVE_GENERATED_DATA:
    if existing_synthetic_files and not ALLOW_OVERWRITE:
        raise FileExistsError(
            "Synthetic-data files already exist:\n"
            + "\n".join(str(path.resolve()) for path in existing_synthetic_files)
            + "\nSet ALLOW_OVERWRITE = True only when replacement is intentional."
        )

    np.save(synthetic_files["features"], X_synthetic)
    np.save(synthetic_files["labels"], y_synthetic)
    synthetic_metadata.to_csv(synthetic_files["metadata"], index=False)
    print("Synthetic arrays and metadata saved to:", SYNTHETIC_DATASET_DIR.resolve())
else:
    missing_saved_files = [
        path for path in synthetic_files.values() if not path.exists()
    ]
    if missing_saved_files:
        raise FileNotFoundError(
            "Validated synthetic files are missing:\n"
            + "\n".join(str(path.resolve()) for path in missing_saved_files)
            + "\nSet SAVE_GENERATED_DATA = True to create them."
        )
    print("Saved synthetic dataset detected; existing files will not be overwritten.")

In [ ]:
if SAVE_GENERATED_DATA:
    SYNTHETIC_OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    similarity_metrics_df.to_csv(
        SYNTHETIC_OUTPUT_DIR
        / "parent_similarity_metrics.csv",
        index=False
    )

    similarity_summary.to_csv(
        SYNTHETIC_OUTPUT_DIR
        / "parent_similarity_summary.csv",
        index=False
    )

    alignment_summary_df.to_csv(
        SYNTHETIC_OUTPUT_DIR
        / "alignment_summary.csv",
        index=False
    )

    shift_effect_df.to_csv(
        SYNTHETIC_OUTPUT_DIR
        / "feature_shift_effect.csv",
        index=False
    )

    distribution_summary_df.to_csv(
        SYNTHETIC_OUTPUT_DIR
        / "distribution_summary.csv",
        index=False
    )

    quality_summary.to_csv(
        SYNTHETIC_OUTPUT_DIR
        / "quality_summary.csv",
        index=False
    )


    generation_manifest = {
        "generation_name": GENERATION_NAME,
        "generation_method": (
            "controlled_signal_domain_augmentation_v1"
        ),
        "description": (
            "Transformation-based synthetic augmentation "
            "derived exclusively from the 10% real "
            "training subset."
        ),
        "random_seed": RANDOM_SEED,
        "source_subset": SOURCE_SUBSET,
        "synthetic_to_real_ratio": SYNTHETIC_RATIO,
        "source_samples": int(
            len(X_real_source)
        ),
        "synthetic_samples": int(
            len(X_synthetic)
        ),
        "synthetic_bird_samples": int(
            np.sum(y_synthetic == 0)
        ),
        "synthetic_drone_samples": int(
            np.sum(y_synthetic == 1)
        ),
        "tensor_shape": list(
            X_synthetic.shape
        ),
        "feature_dtype": str(
            X_synthetic.dtype
        ),
        "label_dtype": str(
            y_synthetic.dtype
        ),
        "value_minimum": float(
            X_synthetic.min()
        ),
        "value_maximum": float(
            X_synthetic.max()
        ),
        "parent_selection": (
            "One synthetic child per source sample"
        ),
        "transformations": {
            "feature_shift": {
                "minimum": -5,
                "maximum": 5,
                "boundary_handling": (
                    "nearest boundary value; no circular wrap"
                )
            },
            "amplitude_scale": {
                "minimum": 0.90,
                "maximum": 1.10
            },
            "contrast_factor": {
                "minimum": 0.90,
                "maximum": 1.10
            },
            "gaussian_noise_standard_deviation": {
                "minimum": 0.002,
                "maximum": 0.015
            },
            "feature_mask": {
                "probability": 0.30,
                "minimum_width": 2,
                "maximum_width": 6,
                "fill_method": "sample mean"
            },
            "final_clipping_interval": [
                0.0,
                1.0
            ]
        },
        "validation": {
            "bird_mean_aligned_correlation": float(
                alignment_summary_df.loc[
                    alignment_summary_df[
                        "parent_target_group"
                    ] == "bird",
                    "aligned_mean_correlation"
                ].iloc[0]
            ),
            "drone_mean_aligned_correlation": float(
                alignment_summary_df.loc[
                    alignment_summary_df[
                        "parent_target_group"
                    ] == "drone",
                    "aligned_mean_correlation"
                ].iloc[0]
            ),
            "bird_mean_aligned_mae": float(
                alignment_summary_df.loc[
                    alignment_summary_df[
                        "parent_target_group"
                    ] == "bird",
                    "aligned_mean_mae"
                ].iloc[0]
            ),
            "drone_mean_aligned_mae": float(
                alignment_summary_df.loc[
                    alignment_summary_df[
                        "parent_target_group"
                    ] == "drone",
                    "aligned_mean_mae"
                ].iloc[0]
            ),
            "samples_below_0.80_aligned_correlation":
                int(
                    np.sum(
                        aligned_correlations < 0.80
                    )
                ),
            "structural_checks_passed": True,
            "distribution_check_passed": True,
            "visual_check_passed": True,
            "pca_overlap_observed": True
        },
        "data_leakage_protection": {
            "parents_from_training_partition_only": True,
            "validation_samples_used": False,
            "test_samples_used": False
        }
    }

    generation_manifest[
        "file_sha256"
    ] = {
        "X_synthetic.npy":
            calculate_sha256(
                synthetic_files["features"]
            ),

        "y_synthetic.npy":
            calculate_sha256(
                synthetic_files["labels"]
            ),

        "metadata_synthetic.csv":
            calculate_sha256(
                synthetic_files["metadata"]
            )
    }

    with open(
        synthetic_files["manifest"],
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(
            generation_manifest,
            file,
            indent=4
        )

    print(
        "Generation manifest saved:",
        synthetic_files[
            "manifest"
        ].resolve()
    )
else:
    with open(synthetic_files["manifest"], "r", encoding="utf-8") as file:
        generation_manifest = json.load(file)
    print("Existing generation manifest retained; no validation artifacts were overwritten.")

### 10.1 Reload and Verify Saved Artifacts

In [ ]:
X_synthetic_saved = np.load(
    synthetic_files["features"],
    mmap_mode="r"
)

y_synthetic_saved = np.load(
    synthetic_files["labels"],
    mmap_mode="r"
)

metadata_synthetic_saved = pd.read_csv(
    synthetic_files["metadata"]
)

with open(
    synthetic_files["manifest"],
    "r",
    encoding="utf-8"
) as file:
    saved_manifest = json.load(file)

assert X_synthetic_saved.shape == (
    1150,
    5,
    150
)

assert y_synthetic_saved.shape == (
    1150,
)

assert len(
    metadata_synthetic_saved
) == 1150

assert np.array_equal(
    np.bincount(y_synthetic_saved),
    [575, 575]
)

assert np.array_equal(
    np.asarray(X_synthetic_saved),
    X_synthetic
)

assert np.array_equal(
    np.asarray(y_synthetic_saved),
    y_synthetic
)

assert (
    calculate_sha256(
        synthetic_files["features"]
    )
    == saved_manifest[
        "file_sha256"
    ]["X_synthetic.npy"]
)

assert (
    calculate_sha256(
        synthetic_files["labels"]
    )
    == saved_manifest[
        "file_sha256"
    ]["y_synthetic.npy"]
)

print(
    "All saved synthetic-data files "
    "were reloaded and verified."
)

## 11. Conclusion — Synthetic-Data Generation and Validation

A balanced transformation-based dataset containing **1,150 synthetic samples**
(575 birds and 575 drones) was generated exclusively from the 10% real training
subset. Every sample is traceable to its parent and recorded transformation
parameters. No validation or test observation participated in generation.

The accepted tensors preserve the required `(5,150)` shape, `float32` type,
`[0,1]` interval, and class balance. Real and synthetic class-level intensity,
variation, and energy statistics are closely matched. Translation-aware mean
parent correlations are 0.9818 for birds and 0.9874 for drones, while aligned
MAE remains low. PCA and paired visual analyses show strong distributional and
structural consistency without an isolated synthetic cluster.

One sample fell slightly below the 0.80 correlation reference because masking
overlapped its dominant peak. Visual inspection confirmed that it remains valid,
so it was retained without data-dependent filtering.

The dataset is approved as the project's first **controlled transformation-based
augmentation baseline**. Notebook 08 will train the unchanged 29,121-parameter
CNN on the 10% real subset combined with these synthetic samples. Its performance
will be compared with the fixed 10% real-only baseline and the 25% real-only
target, using exclusively real validation and test partitions.

### Methodological Limitation

These samples are perturbations of observed real signals and should not be
described as independently simulated radar measurements. Any performance gain
will demonstrate the value of controlled transformation-based augmentation, not
the ability of a generative model to reproduce the full real radar distribution.